# Dependent Gaussian Noise Smoke Test

This notebook checks the reviewer-requested dependent-noise experiment path. Run it from the repository root.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "utility").exists():
    raise RuntimeError(f"Run this notebook from the repository root, not {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

In [ ]:
import numpy as np
import pandas as pd

from utility.data_generator import make_multitarget_regression_dependent_noise

print("Dependent Gaussian generator import: OK")

In [ ]:
target_correlation = 0.6
n_samples = 5000
noise_list = np.array([1.0, 2.0, 3.0])

X, y, coef = make_multitarget_regression_dependent_noise(
    n_samples=n_samples,
    n_features=6,
    n_informative=4,
    n_targets=3,
    noise_list=noise_list,
    correlation=target_correlation,
    correlation_structure="equicorrelated",
    random_state=123,
)

noise = y - np.column_stack([X @ c for c in coef])
empirical_corr = np.corrcoef(noise, rowvar=False)
empirical_std = noise.std(axis=0, ddof=1)

assert X.shape == (n_samples, 6)
assert y.shape == (n_samples, 3)
assert len(coef) == 3
assert np.allclose(empirical_std, noise_list, rtol=0.08)
assert abs(empirical_corr[0, 1] - target_correlation) < 0.05
assert abs(empirical_corr[0, 2] - target_correlation) < 0.05
assert abs(empirical_corr[1, 2] - target_correlation) < 0.05

pd.DataFrame(empirical_corr).round(3)

In [ ]:
from utility.exps import run_abs_res_dependent_gaussian_experiment

result = run_abs_res_dependent_gaussian_experiment(
    dim_list=[10],
    sample_list=[50],
    alpha_list=[0.1],
    trials=200,
    methods=["TSCP_R", "Unscaled", "Bonferroni", "Point_CHR", "Empirical_copula"],
    n_train=int(0.8 * (8000)), n_test=(8000) - int(0.8 * (8000)),
    n_features=5,
    n_informative=5,
    oracle_n_samples=200,
    correlation=0.5,
    correlation_structure="equicorrelated",
)

assert not result.trial_results.empty
assert not result.summary_results.empty
assert set(result.summary_results["method"]) == {"TSCP_R", "Unscaled", "Bonferroni", "Point_CHR", "Empirical_copula"}
assert set(result.summary_results["correlation"]) == {0.5}
assert set(result.summary_results["correlation_structure"]) == {"equicorrelated"}

result.summary_results

If these quick checks pass, the larger reviewer experiment can use `run_abs_res_dependent_gaussian_experiment(...)` with the same dimensions, calibration sizes, alpha values, and method list as the independent Gaussian experiment.